# Factor Pipeline

This is a factor working pipeline from reading to evaluating.

In [ ]:
import os
import dotenv
from pathlib import Path

import pandas as pd

from agent import create
from factool import DuckParquetSource, Evaluator


dotenv.load_dotenv()

## Factor Design

You are able to construction a factor by natural language, but you need to state it in a very specific way. The following block is a example. And then, you are able to copy  the definition text string to generate the factor computing code.

**NOTE:** This notebook is only available for factors that can be calculated in matrix. So if a factor needs complicated calculation or different procession for different stock, DON'T USE THIS BLOCK TO GENERATE FACTOR. Instead, you need to turn to `generate.py` for help.

## 流动性因子

### 月度换手率（monthly_turnover）

**定义**：

月度换手率定义为一个月内每日交易量的总和与每日的流通股数的比值的对数。

**计算步骤**：

1. 提取计算时点t之前的周期T=21天对应时点，记为t+T。
2. 提取数据表quotes_day中t时刻到t+T时刻对应的volume数据矩阵和circulation_a矩阵，t时刻的st和suspended矩阵。
3. 计算每t+T时间内的所有成交量总和与每日流通股数总和：
   $$
   turnover = \frac {\sum_{i=t}^{t+T} volume_i} {\sum_{i=t}^{t+T} circulation\_a_i}
   $$
4. 使用 ~st且 ~suspended数据作为掩码，过滤掉st或suspended的turnover数据，将他们设置为np.nan

In [ ]:
agent = create("code")
prompt = ""  # Copy your full definition text here
await agent.run_streamed(prompt)

Then, you just copy generated code to a new code cell, and run it!

In [ ]:
df = ...

## Optional: Factor Saving

After generating factor, it makes things easier if you save the factor data to disk. Feel free to use `DuckParquetSource` to save any standarized data. `DuckParquetSource.save` helps you with the affairs in saving factor data. There are two available standarized data form: 1. Wide table with different stock codes for each column, and different time lables for each row; 2. Long table with two level index, the first level should be time index, and the second level should be code index. Once you prepare data in the above form, you can pass the data to `DuckParquetSource.save`. Bear in mind that if you use form 1, you need to pass another parameter called `name` to specify the name of the factor, or default name `factor` will be applied.

The `processors` parameter provide a data cleaning way before saving. If set to `None`, `zscore` and `madoutlier` with `dev=5` will be applied

In [ ]:
factor_name = ""  # Apply your factor name here
processors = None
DuckParquetSource(Path(os.getenv("FACTOR_DATA_PATH")) / factor_name).save(
    df, name=factor_name, processors=processors
)

## Factor Evaluation

`Evaluator` is an important component in `factool`. You can initialize it with simply one-line-code. Then you can apply multiple methods to evaluate the factor with different parameters. Note that factor here can be multiple, you can set different factor in a list like `[df1, df2, ...]`

In [ ]:
begin = "2015-01-01"
end = "2025-06-30"
ptype = "close_post"
horizon = 1
skip_horizon = True
ic_method = "spearman"

In [ ]:
price = DuckParquetSource(Path(os.getenv("QUOTESDAY_PATH"))).get_factor(
    ptype, begin=begin, end=end
)
e = Evaluator(factor=df, price=price)

In [ ]:
e.get_info_coef(horizon=horizon, skip_horizon=skip_horizon, method=ic_method)
pd.concat(
    [e.ic, e.ic.rolling(5).mean(), e.ic.cumsum()],
    axis=1,
    key=["IC", "IC rolling 5 mean", "IC cumsum"],
).plot(figsize=(20, 10), secondary_y="IC cumsum")